# Model Evaluation: Semantic Retrieval

This notebook evaluates whether semantic embeddings improve candidate ranking compared to the handcrafted baseline developed in Notebook 2.

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

NUM_CANDIDATES = 500

In [2]:
candidates = []

with open("../data/candidates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        candidates.append(json.loads(line))

        if len(candidates) == NUM_CANDIDATES:
            break

print(f"Loaded {len(candidates)} candidates.")

Loaded 500 candidates.


## Preparing the Job Description
- For smoother document analysing job_description.docx is converted to job_description.txt

In [3]:
with open("../data/job_description.txt", encoding="utf-8") as f:
    job_description = f.read()

## Candidate Representation

Each candidate is converted into a single textual document consisting of:

- Headline
- Summary
- Career History
- Skills

This document is used to generate a semantic embedding.

In [4]:
def build_candidate_document(candidate):

    text = ""

    profile = candidate["profile"]

    text += profile["headline"] + "\n"
    text += profile["summary"] + "\n"

    for job in candidate["career_history"]:
        text += job["title"] + "\n"
        text += job["description"] + "\n"

    for skill in candidate["skills"]:
        text += skill["name"] + " "

    return text

## Experimental Setup

Model:
- all-MiniLM-L6-v2

Similarity Metric:
- Cosine Similarity

Candidate Representation:
- Headline
- Summary
- Career History
- Skills

Evaluation:
- Inspect Top Ranked Candidates

In [5]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)
jd_embedding = model.encode(job_description)

candidate_documents = [
    build_candidate_document(c)
    for c in candidates
]

candidate_embeddings = model.encode(
    candidate_documents,
    batch_size=32,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

- Semantic Similarity:

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

semantic_scores = []

for emb in candidate_embeddings:
    score = cosine_similarity(
        [jd_embedding],
        [emb]
    )[0][0]

    semantic_scores.append(score)

print("Semantic scores generated:", len(semantic_scores))

Semantic scores generated: 500


# DataFrame creation

In [7]:
semantic_df = pd.DataFrame({
    "candidate_id": [
        c["candidate_id"]
        for c in candidates
    ],
    "semantic_score": semantic_scores
})

semantic_df.head()

,candidate_id,semantic_score
0,CAND_0000001,0.481595
1,CAND_0000002,0.550824
2,CAND_0000003,0.514411
3,CAND_0000004,0.485459
4,CAND_0000005,0.426513


- Top semantic matches

In [8]:
semantic_df = semantic_df.sort_values(
    by="semantic_score",
    ascending=False
).reset_index(drop=True)

semantic_df.head(20)

,candidate_id,semantic_score
0,CAND_0000322,0.628581
1,CAND_0000330,0.623160
2,CAND_0000074,0.614951
3,CAND_0000120,0.605683
4,CAND_0000388,0.603567
5,CAND_0000421,0.600074
6,CAND_0000121,0.588077
7,CAND_0000301,0.586341
8,CAND_0000083,0.580252
9,CAND_0000097,0.579076


In [9]:
candidate_lookup = {
    c["candidate_id"]: c
    for c in candidates
}

rows = []

for _, row in semantic_df.head(20).iterrows():

    c = candidate_lookup[row["candidate_id"]]

    rows.append({
        "candidate_id": row["candidate_id"],
        "title": c["profile"]["current_title"],
        "company": c["profile"]["current_company"],
        "experience": c["profile"]["years_of_experience"],
        "semantic_score": round(row["semantic_score"], 3)
    })

pd.DataFrame(rows)

,candidate_id,title,company,experience,semantic_score
0,CAND_0000322,HR Manager,Acme Corp,12.0,0.629
1,CAND_0000330,Civil Engineer,Acme Corp,12.0,0.623
2,CAND_0000074,Operations Manager,Dunder Mifflin,1.9,0.615
3,CAND_0000120,Graphic Designer,Wayne Enterprises,5.7,0.606
4,CAND_0000388,HR Manager,Wipro,6.0,0.604
5,CAND_0000421,Graphic Designer,Globex Inc,4.5,0.600
6,CAND_0000121,Customer Support,Wayne Enterprises,3.7,0.588
7,CAND_0000301,Operations Manager,Dunder Mifflin,1.9,0.586
8,CAND_0000083,Graphic Designer,TCS,6.7,0.580
9,CAND_0000097,Mechanical Engineer,TCS,5.8,0.579


# Evaluation Results

- The semantic retrieval experiment did not produce the expected ranking quality.

- Although sentence embeddings successfully captured general textual similarity, they frequently ranked candidates from unrelated professions (such as HR, Marketing and Civil Engineering) above technically relevant AI candidates.

- This suggests that generic semantic similarity alone is insufficient for recruiter-grade candidate ranking.

- Based on this observation, the final system uses the recruiter-inspired handcrafted ranking model developed in Notebook 2 rather than relying solely on semantic similarity.

## Comparison

| Method                      | Observation                                                |
| --------------------------- | ---------------------------------------------------------- |
| Keyword Matching            | Many false positives                                       |
| Semantic Matching           | Better language understanding but poor recruiter relevance |
| Handcrafted Recruiter Model | Best ranking quality                                       |
